In [2]:
import pickle
import os
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
pkl_path = "cell_profile.pkl"
with open(pkl_path, "rb") as f:
    data = pickle.load(f)

print(f"Cell profile shape: {data.shape}")
data.head()

Cell profile shape: (6975, 41)


,Latitude,Longitude,wind_speed,mean_wind_speed,wind_cf,mean_wind_cf,solar_cf,mean_solar_cf,Province,Zone,...,wind_offshore_fixed_cap,wind_offshore_float_cap,solar_utility_cap,solar_rooftop_cap,fpv_reservoir_area_km2,fpv_natural_area_km2,solar_floating_area,solar_floating_cap,agri_voltaics_area,agri_voltaics_cap
0,23.25,104.75,"[1.4134076400905697, 1.2765993152098343, 1.546...",2.673054,"[0.0, 0.0, 0.0, 0.0, 0.0003230612164566598, 0....",0.027587,"[0.040999999999999995, 0.151, 0.268, 0.3798, 0...",0.169459,Ha_Giang,1,...,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.000000,0.00000
1,23.25,105.00,"[1.161600714490837, 1.0395898467071756, 1.1486...",2.324091,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00481367...",0.016864,"[0.0434, 0.15360000000000001, 0.2646, 0.3694, ...",0.166033,Ha_Giang,1,...,0.0,0.0,34.1535,9.1854,0.0,0.0,0.0,0.0,0.119625,3.58875
2,23.25,105.25,"[1.2220610882332317, 1.033450577598655, 1.0427...",2.280811,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.014128,"[0.0482, 0.16160000000000002, 0.2672, 0.358199...",0.165412,Ha_Giang,1,...,0.0,0.0,122.8776,49.4973,0.0,0.0,0.0,0.0,0.785800,23.57400
3,23.25,105.50,"[2.169471852722621, 1.740391518921886, 1.62100...",2.622165,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.000...",0.026811,"[0.0538, 0.17040000000000002, 0.27080000000000...",0.164791,Ha_Giang,1,...,0.0,0.0,48.7584,27.9603,0.0,0.0,0.0,0.0,0.208800,6.26400
4,23.00,104.75,"[1.092397111469593, 0.9974153493701152, 1.2421...",2.037593,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00638003...",0.007325,"[0.0374, 0.13520000000000001, 0.2412, 0.3442, ...",0.165374,Ha_Giang,1,...,0.0,0.0,19.0059,6.7239,0.0,0.0,0.0,0.0,0.050725,1.52175


In [4]:
# Mapping WP.csv to IBTrACS 

ibtracs_raw = pd.read_csv("ibtracs.WP.list.v04r01.csv", skiprows=[1])
ibtracs_raw.columns = ibtracs_raw.columns.str.strip()
ibtracs_raw = ibtracs_raw.replace(r'^\s*$', np.nan, regex=True)

wp = pd.read_csv("WP.csv")

print(f"IBTrACS raw shape: {ibtracs_raw.shape}")
print(f"WP.csv shape: {wp.shape}")
print(f"\nIBTrACS columns (first 20): {ibtracs_raw.columns.tolist()[:20]}")
print(f"WP columns: {wp.columns.tolist()}")

C:\Users\admin\AppData\Local\Temp\ipykernel_9912\20142727.py:3: DtypeWarning: Columns (19,20,23,24,142,143,144,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  ibtracs_raw = pd.read_csv("ibtracs.WP.list.v04r01.csv", skiprows=[1])
C:\Users\admin\AppData\Local\Temp\ipykernel_9912\20142727.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ibtracs_raw = ibtracs_raw.replace(r'^\s*$', np.nan, regex=True)


IBTrACS raw shape: (246751, 174)
WP.csv shape: (152208, 17)

IBTrACS columns (first 20): ['SID', 'SEASON', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON', 'WMO_WIND', 'WMO_PRES', 'WMO_AGENCY', 'TRACK_TYPE', 'DIST2LAND', 'LANDFALL', 'IFLAG', 'USA_AGENCY', 'USA_ATCF_ID', 'USA_LAT']
WP columns: ['Unnamed: 0', 'number', 'name', 'year', 'month', 'day', 'hour', 'nature', 'lat', 'lon', 'land', 'vmax', 'pressure', 'rmax', 'R34', 'R50', 'R64']


In [5]:
ibtracs_raw.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246751 entries, 0 to 246750
Data columns (total 174 columns):
 #    Column            Dtype  
---   ------            -----  
 0    SID               object 
 1    SEASON            int64  
 2    NUMBER            int64  
 3    BASIN             object 
 4    SUBBASIN          object 
 5    NAME              object 
 6    ISO_TIME          object 
 7    NATURE            object 
 8    LAT               float64
 9    LON               float64
 10   WMO_WIND          object 
 11   WMO_PRES          object 
 12   WMO_AGENCY        object 
 13   TRACK_TYPE        object 
 14   DIST2LAND         int64  
 15   LANDFALL          object 
 16   IFLAG             object 
 17   USA_AGENCY        object 
 18   USA_ATCF_ID       object 
 19   USA_LAT           object 
 20   USA_LON           object 
 21   USA_RECORD        float64
 22   USA_STATUS        object 
 23   USA_WIND          object 
 24   USA_PRES          object 
 25   USA_SSHS          

In [6]:
ibtracs_raw['ISO_TIME'] = pd.to_datetime(ibtracs_raw['ISO_TIME'], errors='coerce')
ibtracs_raw['NAME_CLEAN'] = ibtracs_raw['NAME'].astype(str).str.strip().str.upper()
YEAR_START = 1959
YEAR_END = 2022

ibtracs = ibtracs_raw[
    (ibtracs_raw['SEASON'] >= YEAR_START) &
    (ibtracs_raw['SEASON'] <= YEAR_END) &
    (ibtracs_raw['BASIN'] == 'WP') &
    (ibtracs_raw['TRACK_TYPE'] == 'main')
].copy()

ibtracs = ibtracs.dropna(subset=['ISO_TIME', 'LAT', 'LON'])
ibtracs['year'] = ibtracs['ISO_TIME'].dt.year

print(f"IBTrACS after filtering: {len(ibtracs)} records")
print(f"Unique storms (SID): {ibtracs['SID'].nunique()}")
print(f"Years: {ibtracs['year'].min()} - {ibtracs['year'].max()}")

C:\Users\admin\AppData\Local\Temp\ipykernel_9912\3605303330.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ibtracs_raw['NAME_CLEAN'] = ibtracs_raw['NAME'].astype(str).str.strip().str.upper()


IBTrACS after filtering: 150821 records
Unique storms (SID): 2249
Years: 1959 - 2022


In [7]:
wp['name_clean'] = wp['name'].astype(str).str.strip().str.upper()
wp['datetime'] = pd.to_datetime(
    wp[['year', 'month', 'day']].assign(hour=wp['hour']),
    errors='coerce'
)


wp = wp[(wp['year'] >= YEAR_START) & (wp['year'] <= YEAR_END)].copy()
wp = wp.dropna(subset=['datetime'])

print(f"WP after filtering: {len(wp)} records")
print(f"Unique storm-year combos in WP: {wp.groupby(['name_clean','year']).ngroups}")
print(f"\nWP vmax (m/s) stats:\n{wp['vmax'].describe()}")
print(f"\nWP rmax (km) stats:\n{wp['rmax'].describe()}")

WP after filtering: 152208 records
Unique storm-year combos in WP: 1763

WP vmax (m/s) stats:
count    152208.000000
mean         24.104752
std           9.256926
min          15.261658
25%          18.056444
50%          20.118160
75%          26.307162
max          65.646346
Name: vmax, dtype: float64

WP rmax (km) stats:
count    152208.000000
mean         86.431228
std          33.388414
min           9.260000
25%          66.238666
50%          94.603851
75%         109.999985
max         246.624074
Name: rmax, dtype: float64


In [8]:
ibtracs['match_key'] = ibtracs['NAME_CLEAN'] + '_' + ibtracs['year'].astype(str)
wp['match_key'] = wp['name_clean'] + '_' + wp['year'].astype(str)

# Find common keys
ibtracs_keys = set(ibtracs['match_key'].unique())
wp_keys = set(wp['match_key'].unique())
common_keys = ibtracs_keys & wp_keys

print(f"IBTrACS unique storm-year keys: {len(ibtracs_keys)}")
print(f"WP unique storm-year keys: {len(wp_keys)}")
print(f"Common keys (matched storms): {len(common_keys)}")

# For each matched storm, do a timestamp-based merge (nearest)
merged_records = []

for key in common_keys:
    ib_storm = ibtracs[ibtracs['match_key'] == key].copy()
    wp_storm = wp[wp['match_key'] == key].copy()
    
    ib_storm = ib_storm.sort_values('ISO_TIME').reset_index(drop=True)
    wp_storm = wp_storm.sort_values('datetime').reset_index(drop=True)
    
    # Use merge_asof to match by nearest timestamp
    ib_storm = ib_storm.rename(columns={'ISO_TIME': 'timestamp'})
    wp_storm = wp_storm.rename(columns={'datetime': 'timestamp'})
    
    # merge_asof requires sorted keys
    ib_sorted = ib_storm.sort_values('timestamp')
    wp_sorted = wp_storm[['timestamp', 'vmax', 'pressure', 'rmax', 'R34', 'R50', 'R64']].sort_values('timestamp')
    
    merged = pd.merge_asof(
        ib_sorted,
        wp_sorted,
        on='timestamp',
        tolerance=pd.Timedelta('1.5h'),  # max 1.5 hour difference
        direction='nearest',
        suffixes=('_ib', '_wp')
    )
    
    merged_records.append(merged)

storms_merged = pd.concat(merged_records, ignore_index=True)

storms_merged = storms_merged.rename(columns={'timestamp': 'ISO_TIME'})

print(f"\nMerged records: {len(storms_merged)}")
print(f"Unique storms in merged: {storms_merged['SID'].nunique()}")

IBTrACS unique storm-year keys: 1766
WP unique storm-year keys: 1763
Common keys (matched storms): 1658

Merged records: 129236
Unique storms in merged: 1663


In [9]:
before = len(storms_merged)
storms_merged = storms_merged.dropna(subset=['vmax', 'rmax'])
after = len(storms_merged)
print(f"Dropped {before - after} rows with no WP match for vmax/rmax")
print(f"Remaining: {after} records, {storms_merged['SID'].nunique()} unique storms")

storms_merged = storms_merged[
    (storms_merged['vmax'] >= 0) &
    (storms_merged['rmax'] > 0) &
    (storms_merged['LAT'].notna()) &
    (storms_merged['LON'].notna())
].copy()
print(f"After removing invalid values: {len(storms_merged)} records")

storms_merged = storms_merged.sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)

# Convert vmax from 10m to 100m height using power-law: U(100) = U(10) * (100/10)^0.077 
ALPHA_SHEAR = 0.077
SCALE_10_TO_100 = 10 ** ALPHA_SHEAR  # ≈ 1.194
storms_merged['vmax_100m'] = storms_merged['vmax'] * SCALE_10_TO_100

print(f"Wind speed scaling: U_100m = U_10m * {SCALE_10_TO_100:.4f}")
print(f"\nvmax_10m stats:\n{storms_merged['vmax'].describe()}")
print(f"\nvmax_100m stats:\n{storms_merged['vmax_100m'].describe()}")

# Check time interval
storms_merged['time_diff_h'] = storms_merged.groupby('SID')['ISO_TIME'].diff().dt.total_seconds() / 3600.0

print(f"\nTime interval stats (hours):")
print(storms_merged['time_diff_h'].describe())

# Flag large gaps (> 6 hours)
large_gaps = storms_merged[storms_merged['time_diff_h'] > 6]
print(f"\nRows with time gap > 6h: {len(large_gaps)}")

Dropped 809 rows with no WP match for vmax/rmax
Remaining: 128427 records, 1663 unique storms
After removing invalid values: 128427 records
Wind speed scaling: U_100m = U_10m * 1.1940

vmax_10m stats:
count    128427.000000
mean         24.897959
std           9.693711
min          15.261658
25%          18.202127
50%          20.593556
75%          28.431633
max          65.646346
Name: vmax, dtype: float64

vmax_100m stats:
count    128427.000000
mean         29.727867
std          11.574176
min          18.222238
25%          21.733123
50%          24.588460
75%          33.947032
max          78.380956
Name: vmax_100m, dtype: float64

Time interval stats (hours):
count    126764.000000
mean          2.998509
std           0.317602
min           1.000000
25%           3.000000
50%           3.000000
75%           3.000000
max          84.000000
Name: time_diff_h, dtype: float64

Rows with time gap > 6h: 7


In [11]:
CELL_LAT = "Latitude"
CELL_LON = "Longitude"

VMAX_THRESHOLD_MS = 50 
BUFFER_DEG = 5.0
LAT_MIN = data[CELL_LAT].min() - BUFFER_DEG
LAT_MAX = data[CELL_LAT].max() + BUFFER_DEG
LON_MIN = data[CELL_LON].min() - BUFFER_DEG
LON_MAX = data[CELL_LON].max() + BUFFER_DEG

print(f"Vietnam cell bounds: lat [{data[CELL_LAT].min()}, {data[CELL_LAT].max()}], "
      f"lon [{data[CELL_LON].min()}, {data[CELL_LON].max()}]")
print(f"Search box: lat [{LAT_MIN}, {LAT_MAX}], lon [{LON_MIN}, {LON_MAX}]")

# Filter storm points to search box
storms_nearby = storms_merged[
    (storms_merged['LAT'] >= LAT_MIN) & (storms_merged['LAT'] <= LAT_MAX) &
    (storms_merged['LON'] >= LON_MIN) & (storms_merged['LON'] <= LON_MAX)
].copy()

print(f"\nStorm points in search box: {len(storms_nearby)}")
print(f"Unique storms near Vietnam: {storms_nearby['SID'].nunique()}")

strong_nearby = storms_nearby[storms_nearby['vmax'] > VMAX_THRESHOLD_MS]
print(f"Strong points (vmax > {VMAX_THRESHOLD_MS:.2f} m/s) near Vietnam: {len(strong_nearby)}")
print(f"Unique strong storms near Vietnam: {strong_nearby['SID'].nunique()}")

Vietnam cell bounds: lat [6.25, 23.25], lon [102.25, 112.75]
Search box: lat [1.25, 28.25], lon [97.25, 117.75]

Storm points in search box: 18814
Unique storms near Vietnam: 648
Strong points (vmax > 50.00 m/s) near Vietnam: 86
Unique strong storms near Vietnam: 17


In [12]:
# Interpolate from 3h to 1h: storm track 

def interpolate_storm_track(group, sid):
    group = group.sort_values("ISO_TIME").copy()
    
    if len(group) < 2:
        result = group[['LAT', 'LON', 'vmax', 'vmax_100m', 'rmax', 'R34', 'R50', 'R64', 'ISO_TIME', 'year']].copy()
        result['SID'] = sid
        return result
    
    t0 = group["ISO_TIME"].iloc[0]
    group["t_hours"] = (group["ISO_TIME"] - t0).dt.total_seconds() / 3600.0
    
    t_min = group["t_hours"].min()
    t_max = group["t_hours"].max()
    t_new = np.arange(t_min, t_max + 0.01, 1.0)
    
    result = pd.DataFrame({"t_hours": t_new})
    
    # Interpolate position
    for col in ['LAT', 'LON']:
        f = interp1d(group["t_hours"].values, group[col].values,
                     kind="linear", bounds_error=False, fill_value=(group[col].values[0], group[col].values[-1]))
        result[col] = f(t_new)
    
    # Interpolate intensity — clamp to non-negative (include vmax_100m)
    for col in ['vmax', 'vmax_100m', 'rmax']:
        f = interp1d(group["t_hours"].values, group[col].values,
                     kind="linear", bounds_error=False, fill_value=(group[col].values[0], group[col].values[-1]))
        result[col] = np.maximum(f(t_new), 0.0)
    
    # Interpolate R34, R50, R64
    for col in ['R34', 'R50', 'R64']:
        valid = group.dropna(subset=[col])
        if len(valid) >= 2:
            f = interp1d(valid["t_hours"].values, valid[col].values,
                         kind="linear", bounds_error=False, fill_value=np.nan)
            result[col] = np.maximum(f(t_new), 0.0)
        elif len(valid) == 1:
            result[col] = valid[col].values[0]
        else:
            result[col] = np.nan
    
    result["ISO_TIME"] = t0 + pd.to_timedelta(result["t_hours"], unit="h")
    result["year"] = result["ISO_TIME"].dt.year
    result["SID"] = sid
    
    return result

# Only interpolate storms that have points near Vietnam
affecting_sids = storms_nearby['SID'].unique()
storms_to_interp = storms_merged[storms_merged['SID'].isin(affecting_sids)].copy()

print(f"Interpolating {len(affecting_sids)} storm tracks to 1-hour resolution...")

results = []
for sid, group in storms_to_interp.groupby('SID'):
    results.append(interpolate_storm_track(group, sid))

storms_1h = pd.concat(results, ignore_index=True)
print(f"After interpolation: {len(storms_1h)} records, {storms_1h['SID'].nunique()} storms")

storms_1h_nearby = storms_1h[
    (storms_1h['LAT'] >= LAT_MIN) & (storms_1h['LAT'] <= LAT_MAX) &
    (storms_1h['LON'] >= LON_MIN) & (storms_1h['LON'] <= LON_MAX)
].copy()

print(f"Interpolated points near Vietnam: {len(storms_1h_nearby)}")

Interpolating 648 storm tracks to 1-hour resolution...
After interpolation: 136683 records, 648 storms
Interpolated points near Vietnam: 55766


In [13]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

In [ ]:
# Some wind speeds are negative -> cap to 0 

baseline_speeds_list_before = []
for i in range(len(data)):
    ws = data.iloc[i]['wind_speed']
    ws_array = np.array(ws, dtype=float) if not isinstance(ws, np.ndarray) else np.array(ws, dtype=float)
    baseline_speeds_list_before.append(ws_array)

baseline_speeds_before = np.concatenate(baseline_speeds_list_before)

neg_count_before = np.sum(baseline_speeds_before < 0)
zero_count_before = np.sum(baseline_speeds_before == 0)
pos_count_before = np.sum(baseline_speeds_before > 0)

print(f"\nWind speed distribution (BEFORE capping):")
print(f"  Negative values: {neg_count_before:,} ({100*neg_count_before/len(baseline_speeds_before):.2f}%)")
print(f"  Zero values: {zero_count_before:,} ({100*zero_count_before/len(baseline_speeds_before):.2f}%)")
print(f"  Positive values: {pos_count_before:,} ({100*pos_count_before/len(baseline_speeds_before):.2f}%)")


wind_speed_arrays_capped = []
for i in range(len(data)):
    ws = data.iloc[i]['wind_speed']
    ws_array = np.array(ws, dtype=float)
    ws_array_capped = np.maximum(ws_array, 0.0)  
    wind_speed_arrays_capped.append(ws_array_capped)

data['wind_speed'] = wind_speed_arrays_capped

baseline_speeds_after = np.concatenate(wind_speed_arrays_capped)

neg_count_after = np.sum(baseline_speeds_after < 0)
zero_count_after = np.sum(baseline_speeds_after == 0)
pos_count_after = np.sum(baseline_speeds_after > 0)

print(f"\nWind speed distribution (AFTER capping):")
print(f"  Negative values: {neg_count_after:,} ({100*neg_count_after/len(baseline_speeds_after):.2f}%)")
print(f"  Zero values: {zero_count_after:,} ({100*zero_count_after/len(baseline_speeds_after):.2f}%)")
print(f"  Positive values: {pos_count_after:,} ({100*pos_count_after/len(baseline_speeds_after):.2f}%)")



Data type of wind_speed[0]: <class 'numpy.ndarray'>

Wind speed distribution (BEFORE capping):
  Negative values: 570 (0.00%)
  Zero values: 0 (0.00%)
  Positive values: 61,100,430 (100.00%)

Wind speed distribution (AFTER capping):
  Negative values: 0 (0.00%)
  Zero values: 570 (0.00%)
  Positive values: 61,100,430 (100.00%)


In [32]:
KNOTS_34_MS = 17.5
KNOTS_50_MS = 25.7
KNOTS_64_MS = 32.9

storms_2018 = storms_1h[storms_1h['year'] == 2018].copy()
print(f"Storm points in 2018: {len(storms_2018)}")
print(f"Unique storms in 2018: {storms_2018['SID'].nunique()}")

year_2018_start = pd.Timestamp('2018-01-01 00:00:00')
year_2018_end = pd.Timestamp('2018-12-31 23:00:00')
hours_2018 = pd.date_range(year_2018_start, year_2018_end, freq='1h')
print(f"2018 has {len(hours_2018)} hours")

time_to_hour_idx = {t: i for i, t in enumerate(hours_2018)}

CELL_LAT = "Latitude"
CELL_LON = "Longitude"
cell_lats = data[CELL_LAT].values
cell_lons = data[CELL_LON].values
n_cells = len(data)

print(f"Processing {n_cells} cells for typhoon wind speed overlay...")

if 'wind_speed' not in data.columns:
    raise ValueError("'wind_speed' column not found in data. Please ensure wind speed arrays exist.")
wind_speed_arrays = []
for i in range(n_cells):
    ws = data.iloc[i]['wind_speed']
    ws_array = np.array(ws, dtype=float) if not isinstance(ws, np.ndarray) else ws.astype(float)
    wind_speed_arrays.append(ws_array)

print(f"Loaded baseline wind speed arrays for {n_cells} cells")

total_storms = len(storms_2018)
typhoon_hits = 0
for idx, storm_point in enumerate(storms_2018.itertuples(index=False)):
    if idx % 500 == 0:
        print(f"  Processing storm point {idx}/{total_storms}...")
    
    storm_lat = storm_point.LAT
    storm_lon = storm_point.LON
    storm_time = storm_point.ISO_TIME
    
    hour_idx = time_to_hour_idx.get(storm_time)
    if hour_idx is None:
        continue 
    
    # Determine which radius and wind speed to use (priority: R34 > R50 > R64 > rmax)
    if pd.notna(storm_point.R34) and storm_point.R34 > 0:
        hit_radius = storm_point.R34
        wind_speed = KNOTS_34_MS
        radius_type = "R34"
    elif pd.notna(storm_point.R50) and storm_point.R50 > 0:
        hit_radius = storm_point.R50
        wind_speed = KNOTS_50_MS
        radius_type = "R50"
    elif pd.notna(storm_point.R64) and storm_point.R64 > 0:
        hit_radius = storm_point.R64
        wind_speed = KNOTS_64_MS
        radius_type = "R64"
    else:
        hit_radius = storm_point.rmax
        wind_speed = storm_point.vmax_100m
        radius_type = "rmax"
    
    dist = haversine_km(cell_lats, cell_lons, storm_lat, storm_lon)
    
    # Find cells that are hit (distance < radius)
    hit_cells = np.where(dist < hit_radius)[0]
    
    for cell_idx in hit_cells:
        wind_speed_arrays[cell_idx][hour_idx] = wind_speed
        typhoon_hits += 1

data['wind_speed_typhoon'] = wind_speed_arrays

print(f"  - {n_cells} cells with wind speed arrays (8760 hours each)")
print(f"  - Baseline wind speeds preserved; typhoon hours replaced")
print(f"  - Total typhoon replacements: {typhoon_hits}")
print(f"\nColumn shape per cell: {data.iloc[0]['wind_speed_typhoon'].shape}")


all_speeds = np.concatenate(wind_speed_arrays)
print(f"Overall wind speed range (m/s): [{all_speeds.min():.2f}, {all_speeds.max():.2f}]")
print(f"Overall wind speed mean (m/s): {all_speeds.mean():.2f}")
print(f"Overall wind speed std (m/s): {all_speeds.std():.2f}")

Storm points in 2018: 2651
Unique storms in 2018: 12
2018 has 8760 hours
Processing 6975 cells for typhoon wind speed overlay...
Loaded baseline wind speed arrays for 6975 cells
  Processing storm point 0/2651...
  Processing storm point 500/2651...
  Processing storm point 1000/2651...
  Processing storm point 1500/2651...
  Processing storm point 2000/2651...
  Processing storm point 2500/2651...
  - 6975 cells with wind speed arrays (8760 hours each)
  - Baseline wind speeds preserved; typhoon hours replaced
  - Total typhoon replacements: 31683

Column shape per cell: (8760,)
Overall wind speed range (m/s): [0.00, 24.00]
Overall wind speed mean (m/s): 5.75
Overall wind speed std (m/s): 3.15


In [33]:
data['wind_speed_typhoon']

0       [1.4134076400905697, 1.2765993152098343, 1.546...
1       [1.161600714490837, 1.0395898467071756, 1.1486...
2       [1.2220610882332317, 1.033450577598655, 1.0427...
3       [2.169471852722621, 1.740391518921886, 1.62100...
4       [1.092397111469593, 0.9974153493701152, 1.2421...
                              ...                        
6970    [10.480575988725922, 10.624011656339201, 11.55...
6971    [10.204901616888836, 10.274631049183135, 11.24...
6972    [9.927605282952223, 9.924497087382981, 10.9339...
6973    [9.648147793639797, 9.573244306449926, 10.6259...
6974    [9.372584327606013, 9.22853887982522, 10.32644...
Name: wind_speed_typhoon, Length: 6975, dtype: object

In [34]:
data['wind_speed']

0       [1.4134076400905697, 1.2765993152098343, 1.546...
1       [1.161600714490837, 1.0395898467071756, 1.1486...
2       [1.2220610882332317, 1.033450577598655, 1.0427...
3       [2.169471852722621, 1.740391518921886, 1.62100...
4       [1.092397111469593, 0.9974153493701152, 1.2421...
                              ...                        
6970    [10.480575988725922, 10.624011656339201, 11.55...
6971    [10.204901616888836, 10.274631049183135, 11.24...
6972    [9.927605282952223, 9.924497087382981, 10.9339...
6973    [9.648147793639797, 9.573244306449926, 10.6259...
6974    [9.372584327606013, 9.22853887982522, 10.32644...
Name: wind_speed, Length: 6975, dtype: object